# Drawing with a Turtle: From Single Steps to Functions

A *turtle* is a tiny robot that carries a pen. It knows only two things: **where it is**
and **which way it is facing**. You give it short commands — go forward, turn left — and
it leaves a trail behind it.

That is a deliberately small vocabulary, and that is the point. This notebook uses it to
walk through the way almost every program grows:

| Step | Idea | Section |
|---|---|---|
| 1 | A single instruction | Draw a line |
| 2 | A sequence of instructions | Two intersecting lines |
| 3 | Repeat instead of retype | Loops → triangle, square, pentagon |
| 4 | Name a sequence | Functions for each shape |
| 5 | Add parameters | One function for *every* polygon |
| 6 | Combine functions | A flower and a teddy bear |
| 7 | A function that calls itself | A binary tree |
| 8 | Generalize the recursion | Trees with any branching factor |

Nothing here is about graphics. It is about **decomposition**: noticing repetition,
giving it a name, and giving that name knobs to turn.

## 0. Setup

We use the [`jupyturtle`](https://pypi.org/project/jupyturtle/) package, which draws SVG
graphics directly in a notebook cell. Run the install line **once** if the import fails.

In [1]:
# Run this once if the import in the next cell fails:
# %pip install jupyturtle

import math
import jupyturtle as t

print("jupyturtle ready")

jupyturtle ready


### How the canvas works

Three facts that explain everything you will see below:

- `t.make_turtle(width=W, height=H)` creates a **new** canvas and puts the turtle in the
  **middle** of it, at `(W/2, H/2)`, facing **right**.
- Coordinates are *screen* coordinates: `x` grows to the right, and **`y` grows
  downward** — `(0, 0)` is the top-left corner, not the bottom-left.
- Angles follow the same convention: `t.set_heading(0)` points right, `90` points
  **down**, `180` points left, `270` (or `-90`) points **up**.
  `t.left()` and `t.right()` always turn the way they look on screen.

The commands we need in this notebook:

| Command | Meaning |
|---|---|
| `t.forward(d)` / `t.back(d)` | move `d` pixels along the current heading, drawing if the pen is down |
| `t.left(a)` / `t.right(a)` | turn `a` degrees in place |
| `t.jump_to(x, y)` | teleport **without** drawing |
| `t.set_heading(a)` | face an absolute direction |
| `t.pen_up()` / `t.pen_down()` | stop / resume drawing |
| `t.set_color(c)` / `t.set_width(w)` | change the pen |
| `t.hide()` / `t.show()` | hide or show the turtle marker |

Each drawing must start with `t.make_turtle(...)` **in the same cell** as the commands
that follow it — that is the cell the picture appears in.

## 1. A single instruction: draw a line

The smallest possible program: one command.

In [2]:
t.make_turtle(width=600, height=200)

t.forward(200)   # that's it — one instruction, one line

The turtle started in the middle of the canvas at `(300, 100)` facing right, so the line
runs from `(300, 100)` to `(500, 100)`. The little marker at the end is the turtle itself.

To control *where* the line starts, teleport first with `jump_to`, then set the direction.

In [3]:
t.make_turtle(width=600, height=200)

t.jump_to(50, 100)     # teleport to the left edge — no trail is drawn
t.set_heading(0)       # face right
t.forward(500)         # one long line
t.hide()               # hide the turtle marker for a clean picture

The command from the course introduction is the same idea with two turns mixed in — three
short lines that happen to form three sides of a square:

In [4]:
t.make_turtle(width=800, height=600)

t.forward(20)
t.left(90)
t.forward(20)
t.left(90)
t.forward(20)

Notice how small that drawing looks on an 800 x 600 canvas. **Match the canvas to the
drawing**, not the other way around — a 200-pixel figure needs roughly a 300-pixel canvas.

## 2. A sequence: two intersecting lines

Instructions run top to bottom, and the turtle carries its state from one line to the
next. To draw two *separate* strokes we lift the pen (or use `jump_to`, which never
draws) between them.

In [5]:
t.make_turtle(width=600, height=300)

# first line: horizontal, straight through the middle
t.jump_to(100, 150)
t.set_heading(0)          # 0 degrees = pointing right
t.forward(400)

# second line: diagonal, crossing the first one
t.jump_to(200, 50)
t.set_heading(45)         # 45 degrees = down-and-right (remember: y grows downward)
t.forward(280)

t.hide()

The two lines cross near `(300, 150)`. Change `t.set_heading(45)` to `-45` and re-run the
cell: the second line now goes *up* and to the right, and the crossing point moves. This
is the fastest way to build intuition — change one number, run, look.

### The same X, drawn without lifting the pen

The same figure can be drawn as one continuous path by walking back along a stroke you
already drew. This wastes motion but uses fewer concepts, and it introduces `back()`:

In [6]:
t.make_turtle(width=600, height=300)

t.jump_to(300, 150)       # start at the crossing point

t.set_heading(45)
t.forward(140)            # out along one diagonal
t.back(140)               # and back to the middle

t.set_heading(135)
t.forward(140)
t.back(140)

t.set_heading(225)
t.forward(140)
t.back(140)

t.set_heading(315)
t.forward(140)
t.back(140)

t.hide()

Look at that cell for a moment. Four blocks, identical except for one number. That
repetition is a **signal**, and the next section is the response to it.

## 3. Repetition: a loop instead of copy-paste

A triangle is "forward, turn, forward, turn, forward, turn". Written out by hand:

```python
t.forward(200); t.left(120)
t.forward(200); t.left(120)
t.forward(200); t.left(120)
```

Written as a loop, the *structure* becomes visible: do this pair of steps 3 times.

In [7]:
t.make_turtle(width=600, height=300)

t.jump_to(200, 220)
t.set_heading(0)

for _ in range(3):        # `_` is a conventional name for "I don't use this value"
    t.forward(200)
    t.left(120)           # 360 / 3 = 120

t.hide()

### Why 120 degrees?

At every corner the turtle turns by the **exterior angle**. Walking all the way around a
closed shape and ending up facing the original direction means the turns add up to a full
circle:

$$n \times \text{turn} = 360^\circ \qquad \Longrightarrow \qquad \text{turn} = \frac{360^\circ}{n}$$

So a triangle turns 120°, a square 90°, a pentagon 72°, a hexagon 60°. **Only two numbers
change between all of these**: the repeat count and the turn.

In [8]:
t.make_turtle(width=600, height=250)

# a square: 4 sides, 360 / 4 = 90 degrees
t.jump_to(120, 60)
t.set_heading(0)
for _ in range(4):
    t.forward(130)
    t.left(90)

# a pentagon: 5 sides, 360 / 5 = 72 degrees
t.jump_to(340, 190)
t.set_heading(0)
for _ in range(5):
    t.forward(110)
    t.left(72)

t.hide()

### One loop inside another

If the number of sides is itself a variable, a second loop can draw the whole family at
once. The outer loop walks over the shapes; the inner loop draws one shape.

In [9]:
t.make_turtle(animate=False, width=700, height=220)

x = 90
for n in (3, 4, 5, 6, 8):          # sides
    t.jump_to(x, 180)
    t.set_heading(0)
    for _ in range(n):             # draw one polygon
        t.forward(45)
        t.left(360 / n)
    x = x + 125                    # shift right for the next shape

t.hide()
t.draw()                           # animate=False means: draw once, at the end

> **A note on `animate=False`.** By default the turtle redraws the canvas after *every*
> command, with a short pause — nice for watching, slow for anything large. Passing
> `animate=False` turns the pauses off; you then call `t.draw()` once at the end to show
> the finished picture. Every drawing from here on uses this pattern.

## 4. Naming a sequence: functions

The inner loop above is a complete idea — *draw a square* — buried inside a cell. A
function gives that idea a **name** and a **parameter**, so it can be used anywhere
without being re-read or re-typed.

In [10]:
def triangle(size):
    """Draw an equilateral triangle with sides of `size` pixels."""
    for _ in range(3):
        t.forward(size)
        t.left(120)


def square(size):
    """Draw a square with sides of `size` pixels."""
    for _ in range(4):
        t.forward(size)
        t.left(90)


def pentagon(size):
    """Draw a regular pentagon."""
    for _ in range(5):
        t.forward(size)
        t.left(72)


def hexagon(size):
    """Draw a regular hexagon."""
    for _ in range(6):
        t.forward(size)
        t.left(60)

Defining a function draws nothing. It only teaches Python a new word — the drawing
happens when we **call** it.

One more small function makes positioning readable. `goto` is not a shape; it is a
convenience that packages "teleport, then face this way":

In [11]:
def goto(x, y, heading=0):
    """Move to (x, y) and face `heading`, without drawing anything."""
    t.jump_to(x, y)
    t.set_heading(heading)

In [12]:
t.make_turtle(animate=False, width=700, height=260)

goto(60, 200);  triangle(90)
goto(200, 200); square(90)
goto(340, 200); pentagon(75)
goto(500, 200); hexagon(65)

t.hide()
t.draw()

Compare that cell with the nested-loop cell in section 3. Both draw four shapes, but this
one **says what it draws**. That readability is the whole return on defining a function.

## 5. Parameters: one function for every polygon

Look at the four definitions again. They differ in exactly two places — the repeat count
and the turn — and those two are not independent: the turn is always `360 / n`. So there
is really only **one** number, and it can be a parameter.

In [13]:
def polygon(n, size):
    """Draw a closed regular polygon with `n` equal sides of `size` pixels.

    The turtle ends where it started, facing the direction it started in.
    """
    for _ in range(n):
        t.forward(size)
        t.left(360 / n)

This single function replaces all four of the earlier ones — and every polygon we never
bothered to write:

In [14]:
def triangle(size): polygon(3, size)
def square(size):   polygon(4, size)
def pentagon(size): polygon(5, size)
def hexagon(size):  polygon(6, size)
def octagon(size):  polygon(8, size)

In [15]:
t.make_turtle(animate=False, width=700, height=340)

# nine polygons, from a triangle up to an 11-gon, all with the same side length
for i, n in enumerate(range(3, 12)):
    row, col = divmod(i, 5)               # 5 shapes per row
    goto(110 + col * 115, 120 + row * 180)
    polygon(n, 30)

t.hide()
t.draw()

Notice the side length is fixed at 30 while the shapes grow: more sides of the same length
means a bigger perimeter, so the figure gets larger. To keep the *size* constant instead,
you would shorten the sides as `n` grows — see the exercises.

### Where a polygon becomes a circle

Push `n` up and the sides get too short to see. A 36-sided polygon is, for drawing
purposes, a circle. That gives us a circle function *for free* — and the arithmetic is
just the perimeter of a circle divided into `n` pieces:

In [16]:
def circle(radius, sides=36):
    """Approximate a circle of the given radius with a many-sided polygon.

    The circle is drawn to the LEFT of the turtle, so its center is `radius`
    pixels above the starting point when the turtle faces right.
    """
    side = 2 * math.pi * radius / sides
    polygon(sides, side)


def circle_at(x, y, radius):
    """Draw a circle CENTERED at (x, y)."""
    goto(x, y + radius, 0)     # start below the center, facing right
    circle(radius)

In [17]:
t.make_turtle(animate=False, width=700, height=260)

# the same "circle", drawn with more and more sides
x = 100
for sides in (5, 8, 12, 24, 48):
    goto(x, 190)
    circle(55, sides=sides)
    x = x + 130

t.hide()
t.draw()

## 6. Composition: pictures built from shape functions

With `polygon`, `circle_at`, and `goto` in hand, a picture is no longer a wall of turtle
commands — it is a short list of named parts. Two examples.

### 6a. A flower

A flower head is one polygon drawn over and over, each copy rotated a little. That is a
loop whose body is a function call:

In [18]:
def flower_head(petals=12, size=42, sides=6):
    """Draw `petals` copies of a polygon, fanned around the current position."""
    for _ in range(petals):
        polygon(sides, size)
        t.left(360 / petals)


def stem(length):
    """Draw a stem upward from the current position, returning to the start."""
    t.set_heading(270)          # 270 degrees = up
    t.forward(length)
    t.back(length)


def leaf(size):
    """A leaf is just a thin four-sided shape."""
    for _ in range(2):
        t.forward(size)
        t.left(60)
        t.forward(size)
        t.left(120)

In [19]:
t.make_turtle(animate=False, width=700, height=520)

# --- ground ---
t.set_color('#8B5E3C')
t.set_width(3)
goto(0, 470, 0)
t.forward(700)

# --- stem ---
t.set_color('#2E7D32')
t.set_width(4)
goto(350, 470)
stem(230)

# --- leaves ---
goto(350, 360, 20)
leaf(70)
goto(350, 300, 160)
leaf(70)

# --- petals ---
t.set_color('#C2185B')
t.set_width(2)
goto(350, 240)
flower_head(petals=12, size=40, sides=6)

# --- center ---
t.set_color('#F9A825')
circle_at(350, 240, 22)

# --- sun ---
t.set_color('#FB8C00')
circle_at(610, 80, 45)

t.hide()
t.draw()

Read the cell top to bottom: ground, stem, leaves, petals, center, sun. Six ideas, six
lines of intent. Every turtle command that actually does the work is hidden inside a
function we wrote earlier — and each of those functions was small enough to check on its
own.

### 6b. A teddy bear

A teddy bear is a pile of circles. Since `circle_at` places a circle by its center, the
whole drawing becomes a list of coordinates and radii:

In [20]:
def teddy_bear():
    """Draw a teddy bear out of circles."""
    t.set_width(2)

    # ears
    t.set_color('#8D6E63')
    circle_at(288, 132, 32)
    circle_at(412, 132, 32)

    # head and body
    t.set_color('#A1887F')
    circle_at(350, 200, 78)
    circle_at(350, 370, 105)

    # arms
    circle_at(238, 330, 44)
    circle_at(462, 330, 44)

    # legs
    circle_at(292, 470, 46)
    circle_at(408, 470, 46)

    # belly and muzzle
    t.set_color('#D7CCC8')
    circle_at(350, 385, 62)
    circle_at(350, 232, 30)

    # eyes and nose
    t.set_color('#3E2723')
    circle_at(326, 186, 8)
    circle_at(374, 186, 8)
    circle_at(350, 218, 10)

In [21]:
t.make_turtle(animate=False, width=700, height=560)

teddy_bear()

t.hide()
t.draw()

`teddy_bear` calls `circle_at`, which calls `circle`, which calls `polygon`, which calls
`t.forward` and `t.left`. Four layers, each one written in terms of the layer below it.
That stack is what people mean by *decomposition*, and it is the reason a fifteen-line
function can draw something a beginner would not attempt with raw `forward`/`left` calls.

## 7. Recursion: a binary tree

Every function so far called *other* functions. A **recursive** function calls **itself**
on a smaller version of the same problem.

A tree is the classic example, because a tree is defined in terms of trees:

> A tree of depth 0 is nothing at all.
> A tree of depth `d` is a trunk, with a smaller tree of depth `d-1` growing to the left
> of its tip, and another growing to the right.

That description translates almost word for word into code:

In [22]:
def binary_tree(length, depth, angle=25, shrink=0.72):
    """Draw a binary tree and return the turtle to where it started.

    length : length of the trunk in pixels
    depth  : how many more levels to draw (0 = stop)
    angle  : how far each branch tilts away from its parent
    shrink : how much shorter each branch is than its parent
    """
    if depth == 0:              # BASE CASE: nothing left to draw
        return

    t.forward(length)           # the trunk

    t.left(angle)                                            # aim left
    binary_tree(length * shrink, depth - 1, angle, shrink)   # the left subtree
    t.right(2 * angle)                                       # swing across to the right
    binary_tree(length * shrink, depth - 1, angle, shrink)   # the right subtree
    t.left(angle)                                            # undo the aiming

    t.back(length)              # walk back down the trunk

Two details do all the work:

- **The base case.** `if depth == 0: return` is what stops the recursion. Without it the
  function would call itself forever and Python would raise `RecursionError`.
- **Leave no trace.** The last three lines undo every turn and the forward move, so the
  turtle finishes exactly where and how it started. That is what makes it safe to call
  twice in a row: the second call begins from a known position.

Start small enough to check by eye.

In [23]:
t.make_turtle(animate=False, width=700, height=300)

goto(350, 270, 270)        # bottom center, facing up
binary_tree(90, 3)         # depth 3: trunk + 2 + 4 branches

t.hide()
t.draw()

In [24]:
t.make_turtle(animate=False, width=700, height=560)

t.set_color('#4E342E')
t.set_width(2)

goto(350, 540, 270)        # bottom center, facing up
binary_tree(115, 9)        # same function, one number changed

t.hide()
t.draw()

The only difference between the last two cells is `3` versus `9`. Depth 9 draws
`2⁹ = 512` twigs and 1023 line segments — none of which we described individually. This
is recursion's payoff: the *description* stays six lines long no matter how large the
output gets.

Change `angle` and `shrink` and re-run to see how much of a tree's character comes from
two numbers:

In [25]:
t.make_turtle(animate=False, width=700, height=340)

t.set_color('#33691E')

settings = [(15, 0.75), (30, 0.72), (50, 0.68)]
x = 130
for angle, shrink in settings:
    goto(x, 320, 270)
    binary_tree(70, 7, angle=angle, shrink=shrink)
    x = x + 220

t.hide()
t.draw()

## 8. Generalizing the recursion: any branching factor

`binary_tree` hard-codes "two branches" the same way `square` hard-coded "four sides". The
fix is the same one as in section 5: turn the constant into a **parameter**.

With `b` branches spread over a total fan of `spread` degrees, the gap between neighboring
branches is `spread / (b - 1)`. The turtle aims at the leftmost branch, then steps across
the fan one gap at a time:

```
        \  |  /          b = 3, spread = 80
         \ | /           aim left 40, then turn right 40 twice
          \|/
           |
```

In [26]:
def tree(length, depth, branching=3, spread=80, shrink=0.62):
    """Draw a tree where every node splits into `branching` branches.

    length    : length of this trunk in pixels
    depth     : levels remaining (0 = stop)
    branching : number of branches at each node (1 to 5 look reasonable)
    spread    : total angle, in degrees, covered by the fan of branches
    shrink    : length ratio between a branch and its parent
    """
    if depth == 0:
        return

    t.forward(length)

    if branching == 1:                      # no fan to build: just keep going
        tree(length * shrink, depth - 1, branching, spread, shrink)
    else:
        step = spread / (branching - 1)     # angle between neighboring branches
        t.left(spread / 2)                  # aim at the leftmost branch
        for i in range(branching):
            tree(length * shrink, depth - 1, branching, spread, shrink)
            if i < branching - 1:
                t.right(step)               # swing over to the next branch
        t.left(spread / 2)                  # net turn so far was `spread / 2` right

    t.back(length)

`binary_tree(90, 5, angle=25)` and `tree(90, 5, branching=2, spread=50)` draw the same
figure — the second one just does not assume the number 2.

Because a node with `b` branches at depth `d` produces `b**d` twigs, the drawing grows
*fast*. Depth is reduced as branching increases so that all four trees stay a comparable
size:

In [27]:
t.make_turtle(animate=False, width=760, height=340)

t.set_color('#1B5E20')

x = 110
for branching, depth in [(2, 6), (3, 5), (4, 4), (5, 4)]:
    goto(x, 320, 270)
    tree(60, depth, branching=branching, spread=90, shrink=0.58)
    x = x + 190

t.hide()
t.draw()

In [28]:
# How many line segments does each of those trees contain?
for branching, depth in [(2, 6), (3, 5), (4, 4), (5, 4)]:
    twigs = branching ** depth
    segments = sum(branching ** k for k in range(depth))
    print(f"branching={branching}, depth={depth}: "
          f"{twigs:>5} twigs, {segments:>5} trunk segments")

branching=2, depth=6:    64 twigs,    63 trunk segments
branching=3, depth=5:   243 twigs,   121 trunk segments
branching=4, depth=4:   256 twigs,    85 trunk segments
branching=5, depth=4:   625 twigs,   156 trunk segments


### One tree, with a little color

A last refinement: use `depth` — a value the function already has — to decide the pen. Thin
green twigs near the top, a thick brown trunk at the bottom, with no extra bookkeeping.

In [29]:
def colored_tree(length, depth, branching=3, spread=70, shrink=0.6):
    """Like `tree`, but the pen thins and turns green toward the twigs."""
    if depth == 0:
        return

    t.set_width(max(1, depth))                          # thick near the trunk
    t.set_color('#4E342E' if depth > 2 else '#2E7D32')  # brown wood, green tips
    t.forward(length)

    if branching == 1:
        colored_tree(length * shrink, depth - 1, branching, spread, shrink)
    else:
        step = spread / (branching - 1)
        t.left(spread / 2)
        for i in range(branching):
            colored_tree(length * shrink, depth - 1, branching, spread, shrink)
            if i < branching - 1:
                t.right(step)
        t.left(spread / 2)

    t.set_width(max(1, depth))                          # restore on the way down
    t.set_color('#4E342E' if depth > 2 else '#2E7D32')
    t.back(length)

In [30]:
t.make_turtle(animate=False, width=700, height=520)

goto(350, 500, 270)
colored_tree(120, 6, branching=3, spread=72, shrink=0.6)

t.hide()
t.draw()

## What just happened

Every section solved the same problem the same way:

1. **Write it out.** Section 2 drew an X with four nearly identical blocks.
2. **Spot the repetition.** The blocks differed by one number.
3. **Loop it.** Section 3 replaced the copies with `for _ in range(n)`.
4. **Name it.** Section 4 wrapped the loop in `square`, `pentagon`, …
5. **Parameterize it.** Section 5 noticed those functions differed by one number too, and
   collapsed them into `polygon(n, size)`.
6. **Compose.** Section 6 built pictures out of named parts instead of turtle commands.
7. **Recurse.** Sections 7 and 8 let a function call itself, so six lines describe a
   thousand-segment tree.

That progression — repetition → loop → function → parameter → composition — is not about
turtles. It is what you will do to a data-cleaning script, a plotting routine, and a
model-fitting pipeline for the rest of the course.

Two habits worth keeping:

- **Leave no trace.** `polygon` ends where it started; `binary_tree` undoes its own turns.
  Functions that restore the state they found are the ones you can call twice without
  thinking.
- **Always have a base case.** Every recursive function needs a condition that stops it.

## Exercises

**1. A star.** A five-pointed star is `polygon`'s stubborn cousin: 5 sides, but the turtle
turns `144` degrees instead of `72`, so it goes around the circle twice. Write
`star(size)`, then generalize it to `star(points, size, skip=2)` where the turn is
`180 - 180 / points * skip`... or just experiment with turn angles until it looks right.

**2. Constant size.** In section 5 all polygons used the same side length, so the shapes grew as
`n` increased. Write `polygon_in_circle(n, radius)` that draws an `n`-gon whose corners lie
on a circle of the given radius. *Hint:* the side length is `2 * radius * sin(pi / n)`.

**3. A spiral.** Modify the `polygon` loop so the side length grows a little on every step
(`size = size * 1.05`) and the turn stays fixed. Try turns of 89, 90, and 91 degrees and
explain the difference.

**4. Your own picture.** Using `goto`, `polygon`, and `circle_at`, draw a house (square +
triangle roof + rectangle door), a robot, or a snowman. Write one function per part, then
one function that calls them all.

**5. An asymmetric tree.** Change `binary_tree` so the left branch is shorter than the
right one (say `0.6` versus `0.8` of the trunk). Real trees are not symmetric.

**6. A random tree.** `import random`, then jitter the angle and the shrink factor inside
`tree` with `random.uniform(...)`. Run the cell several times — no two trees alike.

**7. Count the calls.** Add a global counter that increments every time `tree` is called,
and check it against the `sum(branching ** k ...)` formula above for a few settings.